# Actividad 5: entrenamiento, ajuste y registro con MLflow

Este notebook compara **regresión logística** y **random forest** mediante
GridSearchCV, validación cruzada estratificada y métricas estandarizadas.

In [ ]:
# Instalación de dependencias
!pip -q install "mlflow>=3.0,<4.0" "scikit-learn>=1.7,<2.0" pandas matplotlib joblib

## Opción A: clonar el repositorio desde GitHub

El notebook utiliza el repositorio completo publicado en GitHub.

In [ ]:
REPO_URL = "https://github.com/HamDan314/cancermm.git"

# Descomenta estas líneas para ejecutar directamente desde GitHub.
# !git clone "$REPO_URL"
# %cd cancermm/entrenamiento

# Cuando el ZIP se sube manualmente a Colab, sitúate en su carpeta:
# %cd /content/cancermm/entrenamiento

In [ ]:
from pathlib import Path
import sys

RAIZ = Path.cwd()
if not (RAIZ / "fuentes" / "train.py").exists():
    raise FileNotFoundError(
        "Ubícate en la carpeta cancermm/entrenamiento antes de continuar."
    )

sys.path.insert(0, str(RAIZ / "fuentes"))
print("Raíz del proyecto:", RAIZ)

## Preparación y revisión de los datos

In [ ]:
import pandas as pd
from datos_prep import preparar_archivo

ruta_original = RAIZ / "datos/datos_ini/cancer_mama_original.csv"
ruta_limpia = RAIZ / "datos/datos_limp/cancer_mama_limpio.csv"

df = preparar_archivo(ruta_original, ruta_limpia)
display(df.head())
print("Dimensiones:", df.shape)
print("\nDistribución de clases:")
display(df["diagnostico"].value_counts(normalize=True).rename("proporción"))

## Control de calidad, limpieza y versionado del dataset

En esta sección se documentan los criterios aplicados antes del entrenamiento:

- **Duplicados:** se identifican y eliminan registros completamente repetidos.
- **Valores faltantes:** las variables predictoras se convierten a formato numérico y los faltantes se imputan con la mediana.
- **Estandarización:** se realiza posteriormente dentro del `Pipeline` con `StandardScaler`, evitando fuga de información durante la validación cruzada.
- **Versionado:** se conserva una copia original en `datos_ini` y una copia procesada en `datos_limp`. Además, se genera un manifiesto con fecha, dimensiones y huella SHA-256.


In [ ]:
# Auditoría antes y después de la limpieza
import pandas as pd

df_original = pd.read_csv(ruta_original)

auditoria = pd.DataFrame({
    "criterio": [
        "Filas originales",
        "Columnas",
        "Duplicados originales",
        "Valores faltantes originales",
        "Filas después de limpieza",
        "Duplicados después de limpieza",
        "Valores faltantes después de limpieza"
    ],
    "resultado": [
        df_original.shape[0],
        df_original.shape[1],
        int(df_original.duplicated().sum()),
        int(df_original.isna().sum().sum()),
        df.shape[0],
        int(df.duplicated().sum()),
        int(df.isna().sum().sum())
    ]
})

display(auditoria)

print(
    f"Se eliminaron {df_original.shape[0] - df.shape[0]} filas durante la limpieza."
)


In [ ]:
# Creación del manifiesto de versión del dataset
from datetime import datetime, timezone
from hashlib import sha256
import json

def calcular_sha256(ruta):
    hash_archivo = sha256()
    with open(ruta, "rb") as archivo:
        for bloque in iter(lambda: archivo.read(8192), b""):
            hash_archivo.update(bloque)
    return hash_archivo.hexdigest()

manifiesto = {
    "version_dataset": "1.0.0",
    "fecha_generacion_utc": datetime.now(timezone.utc).isoformat(),
    "archivo_original": str(ruta_original),
    "archivo_limpio": str(ruta_limpia),
    "filas_originales": int(df_original.shape[0]),
    "filas_limpias": int(df.shape[0]),
    "columnas": int(df.shape[1]),
    "duplicados_eliminados": int(df_original.shape[0] - df.shape[0]),
    "valores_faltantes_finales": int(df.isna().sum().sum()),
    "sha256_original": calcular_sha256(ruta_original),
    "sha256_limpio": calcular_sha256(ruta_limpia)
}

ruta_manifiesto = RAIZ / "datos/datos_limp/version_dataset.json"
ruta_manifiesto.parent.mkdir(parents=True, exist_ok=True)

with open(ruta_manifiesto, "w", encoding="utf-8") as archivo:
    json.dump(manifiesto, archivo, indent=2, ensure_ascii=False)

display(pd.Series(manifiesto, name="valor").to_frame())
print("Manifiesto guardado en:", ruta_manifiesto)


## Análisis exploratorio: patrones y anomalías

Las siguientes visualizaciones se utilizan para comprobar la distribución de clases, detectar valores extremos y analizar relaciones entre variables. Los valores atípicos no se eliminan automáticamente, porque en un problema médico pueden representar casos reales relevantes.


In [ ]:
# 1. Distribución de la variable objetivo
import matplotlib.pyplot as plt

conteo_clases = (
    df["diagnostico"]
    .map({0: "Maligno", 1: "Benigno"})
    .value_counts()
    .reindex(["Maligno", "Benigno"])
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(conteo_clases.index, conteo_clases.values)
ax.set_title("Distribución de clases")
ax.set_xlabel("Diagnóstico")
ax.set_ylabel("Número de registros")

for indice, valor in enumerate(conteo_clases.values):
    ax.text(indice, valor, str(valor), ha="center", va="bottom")

plt.tight_layout()
plt.show()

proporcion_mayoritaria = conteo_clases.max() / conteo_clases.sum()
print(
    f"La clase mayoritaria representa {proporcion_mayoritaria:.1%} del dataset. "
    "Existe un desbalance moderado, por lo que se utiliza StratifiedKFold."
)


In [ ]:
# 2. Boxplots para identificar posibles valores atípicos
variables_boxplot = [
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean area",
    "mean smoothness"
]

fig, ax = plt.subplots(figsize=(11, 5))
ax.boxplot(
    [df[columna].dropna() for columna in variables_boxplot],
    tick_labels=variables_boxplot,
    showfliers=True
)
ax.set_title("Distribución y posibles valores atípicos")
ax.set_ylabel("Valor original de la variable")
ax.tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.show()

q1 = df[variables_boxplot].quantile(0.25)
q3 = df[variables_boxplot].quantile(0.75)
iqr = q3 - q1

atipicos = (
    (df[variables_boxplot] < (q1 - 1.5 * iqr))
    | (df[variables_boxplot] > (q3 + 1.5 * iqr))
).sum().sort_values(ascending=False)

display(atipicos.rename("cantidad_atipicos_IQR").to_frame())

print(
    "Los puntos extremos se conservan porque pueden corresponder a tumores con "
    "características clínicas reales. La estandarización reducirá diferencias "
    "de escala, pero no borrará información relevante."
)


In [ ]:
# 3. Correlaciones entre variables numéricas
correlaciones = df.drop(columns=["diagnostico"]).corr()

# Selección de las 10 variables con mayor relación absoluta con el diagnóstico
correlacion_objetivo = (
    df.corr(numeric_only=True)["diagnostico"]
    .drop("diagnostico")
    .abs()
    .sort_values(ascending=False)
)

variables_correlacion = correlacion_objetivo.head(10).index.tolist()
matriz_reducida = df[variables_correlacion].corr()

fig, ax = plt.subplots(figsize=(10, 8))
imagen = ax.imshow(matriz_reducida, aspect="auto", vmin=-1, vmax=1)
ax.set_xticks(range(len(variables_correlacion)))
ax.set_yticks(range(len(variables_correlacion)))
ax.set_xticklabels(variables_correlacion, rotation=90)
ax.set_yticklabels(variables_correlacion)
ax.set_title("Correlación de las variables más relacionadas con el diagnóstico")
fig.colorbar(imagen, ax=ax, label="Correlación")

plt.tight_layout()
plt.show()

display(
    correlacion_objetivo.head(10)
    .rename("correlacion_absoluta_con_diagnostico")
    .to_frame()
)

print(
    "Las variables relacionadas con radio, perímetro y área presentan alta "
    "correlación entre sí. Esto indica posible redundancia, pero se conservan "
    "para comparar un modelo lineal con un modelo basado en árboles."
)


In [ ]:
# 4. Relación entre dos variables relevantes y la clase
fig, ax = plt.subplots(figsize=(8, 5))

for clase, etiqueta in [(0, "Maligno"), (1, "Benigno")]:
    subconjunto = df[df["diagnostico"] == clase]
    ax.scatter(
        subconjunto["mean radius"],
        subconjunto["mean texture"],
        label=etiqueta,
        alpha=0.65
    )

ax.set_title("Relación entre radio medio y textura media")
ax.set_xlabel("Radio medio")
ax.set_ylabel("Textura media")
ax.legend()

plt.tight_layout()
plt.show()

print(
    "Se observa una separación parcial entre las clases. Los casos malignos "
    "tienden a concentrarse en valores mayores de radio, aunque existe "
    "superposición; por ello resulta útil comparar regresión logística y "
    "random forest."
)


### Interpretación del análisis exploratorio

1. La distribución de clases presenta un desbalance moderado, no extremo. Por eso se emplea una división estratificada y `StratifiedKFold`.
2. Los boxplots muestran observaciones alejadas de los cuartiles. Se mantienen porque podrían ser casos clínicos válidos y no errores de captura.
3. Las variables geométricas, especialmente radio, perímetro y área, están fuertemente correlacionadas. Esto puede afectar la interpretación de los coeficientes de la regresión logística, aunque no impide su capacidad predictiva.
4. La gráfica de dispersión muestra separación parcial y superposición entre diagnósticos. Esta situación justifica comparar un modelo lineal con random forest, que puede capturar relaciones no lineales.
5. La estandarización se ejecuta dentro del pipeline y dentro de cada fold de validación cruzada, evitando utilizar información del conjunto de prueba durante el ajuste.


## Ejecución del entrenamiento y registro

In [ ]:
# Ejecuta ambos GridSearchCV y registra los experimentos.
!python fuentes/train.py

## Comparación de resultados

In [ ]:
resultados = pd.read_csv(RAIZ / "resultados/comparacion_modelos.csv")
columnas = [
    "modelo", "accuracy_test", "precision_test", "recall_test",
    "f1_test", "roc_auc_test", "mejor_f1_cv",
    "tiempo_entrenamiento_seg"
]
display(resultados[columnas].sort_values("f1_test", ascending=False))

## Visualización de MLflow dentro de Colab

In [ ]:
# En una ejecución local:
# !mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000

# En Colab, una opción sencilla es revisar directamente los resultados
# guardados y descargar la carpeta mlruns para abrirla después localmente.
!zip -qr Actividad5_resultados.zip resultados mlruns mlflow.db
print("Archivo creado: Actividad5_resultados.zip")

## Conclusiones orientativas

- El modelo ganador debe elegirse con base en el F1-score de prueba, sin ignorar
  recall, precision y ROC-AUC.
- La regresión logística ofrece mayor interpretabilidad y normalmente menor
  costo computacional.
- Random forest puede capturar relaciones no lineales, aunque suele requerir
  más tiempo y memoria.
- La diferencia entre F1 de validación cruzada y F1 de prueba ayuda a detectar
  posible sobreajuste.

## Resultados validados incluidos en el repositorio

La ejecución reproducible incluida en esta entrega obtuvo:

- **Regresión logística:** Accuracy 0.9825, Precision 0.9861, Recall 0.9861, F1 0.9861 y ROC-AUC 0.9960.
- **Random forest:** Accuracy 0.9474, Precision 0.9583, Recall 0.9583, F1 0.9583 y ROC-AUC 0.9939.

La regresión logística fue seleccionada como mejor modelo por su mayor F1-score de prueba y su desempeño estable en validación cruzada. Los valores completos se encuentran en `resultados/comparacion_modelos.csv`.